# Langfuse 101: understanding an LLM application

An LLM application can return an answer while hiding the information we need to understand it. What did the model receive? Which step was slow? Was the answer correct? Did a new prompt improve the result?

Langfuse connects these questions to recorded evidence. This notebook follows one small support bot from its first trace through debugging, prompt versioning, quality scores, experiments, and trace analysis with Claude through MCP.

The bot has two steps: `read_policy` supplies a fictional refund policy, and `answer` uses that policy to respond. Each successful request makes one model call. A hardcoded policy keeps the example easy to follow; there is no database or vector search.

The sections build on one another and run from top to bottom. Model responses may vary, so the explanations describe the evidence to look for rather than promising exact wording.

## 1. Connecting the application to Langfuse

### Dependencies and credentials

This notebook uses Python 3.11 or newer; Python 3.12 is a suitable choice. It needs a Langfuse project with its public and secret keys, and an OpenAI API key with model access and available credit. A separate demo project keeps these examples easy to identify.

In [ ]:
%pip install -q langfuse==4.17.0 langgraph==1.2.14 langchain==1.4.3 langchain-openai==1.6.7


### A project and a tracing callback

The Langfuse base URL must match the project's region: `https://cloud.langfuse.com` for EU or `https://us.cloud.langfuse.com` for US.

`get_client()` creates the Langfuse client, and `auth_check()` verifies access to the project. `CallbackHandler()` connects LangChain and LangGraph execution events to Langfuse. A unique run identifier and session identifier let us associate the examples from this execution.

The `development` environment identifies where the telemetry came from. It is a filterable attribute; it does not change the application's behavior.

In [ ]:
import os
from getpass import getpass
from datetime import datetime, timezone

for name in ("LANGFUSE_PUBLIC_KEY", "LANGFUSE_SECRET_KEY", "OPENAI_API_KEY"):
    if not os.environ.get(name):
        os.environ[name] = getpass(f"{name}: ")
os.environ["LANGFUSE_BASE_URL"] = input("Langfuse base URL [https://cloud.langfuse.com]: ").strip() or "https://cloud.langfuse.com"
os.environ["LANGFUSE_TRACING_ENVIRONMENT"] = "development"

from langfuse import get_client, propagate_attributes
from langfuse.langchain import CallbackHandler

langfuse = get_client()
assert langfuse.auth_check(), "Check both Langfuse keys and the project's region."
handler = CallbackHandler()
RUN = datetime.now(timezone.utc).strftime("%Y%m%d-%H%M%S")
SESSION = f"hari-demo-{RUN}"
MODEL = "gpt-4o-mini"  # Replace with a chat model available to your account if needed.
print("Ready. Demo session:", SESSION)


## 2. A small application and its trace structure

### The support graph

Our fictional policy allows refunds within **30 days of purchase**. The graph passes this policy from `read_policy` to `answer`, where a prompt and a model produce the response.

The state holds the question, policy, and answer. `POLICY`, `DELAY`, and `FAIL_POLICY` are simple controls we will change later to create a wrong answer, a slow step, and a technical failure.

### Traces, observations, spans, and generations

A **trace** groups the recorded work for one request. An **observation** describes a unit of work within that trace. **Spans** capture application or framework steps; **generations** capture model calls, including model inputs, outputs, and available usage information.

For this graph, the trace contains the policy step and answer step, with a model generation nested inside the answer workflow. Framework and prompt observations may appear too. Parent-child relationships show how the work fits together.

The `answer` function passes LangGraph's `config` into `chain.invoke()`. This carries the tracing callback into the nested model call, so the trace includes the model activity as well as the graph steps.

In [ ]:
from typing import TypedDict
from time import sleep
from langgraph.graph import StateGraph, START, END
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate

POLICY = "Refunds are available within 30 days of purchase."
DELAY = 0
FAIL_POLICY = False
TEMPLATE = "Answer using only this policy: {policy}\nQuestion: {question}"
llm = ChatOpenAI(model=MODEL, temperature=0, max_tokens=120, max_retries=0, timeout=30)
chain = ChatPromptTemplate.from_template(TEMPLATE) | llm

class State(TypedDict):
    question: str
    policy: str
    answer: str

def read_policy(state):
    sleep(DELAY)
    if FAIL_POLICY:
        raise RuntimeError("Demo: policy service unavailable")
    return {"policy": POLICY}

def answer(state, config):
    response = chain.invoke(
        {"question": state["question"], "policy": state["policy"]}, config=config
    )
    return {"answer": response.content}

builder = StateGraph(State)
builder.add_node("read_policy", read_policy)
builder.add_node("answer", answer)
builder.add_edge(START, "read_policy")
builder.add_edge("read_policy", "answer")
builder.add_edge("answer", END)
graph = builder.compile()


## 3. Recording the first request

### Automatic tracing and request context

The `run_demo()` helper runs the graph with the callback attached. It gives each request a name and adds a user ID, session ID, tags, and metadata. These attributes make the recorded requests easier to find and compare.

The SDK buffers telemetry before sending it. `flush()` sends pending events, including when the graph fails. It is useful in notebooks and other short-lived executions that might finish before a background export completes.

These requests run sequentially, so `handler.last_trace_id` identifies the most recent trace. Concurrent applications need an explicit request trace context rather than relying on a shared handler's last ID.

In [ ]:
def run_demo(question, scenario):
    with propagate_attributes(
        trace_name=f"support-{scenario}", user_id="demo-user", session_id=SESSION,
        tags=["langfuse-101", scenario], metadata={"demo_run": RUN, "scenario": scenario},
    ):
        try:
            result = graph.invoke(
                {"question": question},
                config={"callbacks": [handler], "run_name": f"support-{scenario}"},
            )
            return result["answer"], handler.last_trace_id
        finally:
            langfuse.flush()

baseline_answer, baseline_id = run_demo("What is the refund window?", "baseline")
print(baseline_answer)
print("Trace ID:", baseline_id)


## Inspecting recorded observations in the notebook

In [ ]:
import json

observations = langfuse.api.observations.get_many(
    trace_id=baseline_id,
    fields="core,basic,io,model,usage,metrics",
    limit=100,
)

print(json.dumps(
    [observation.dict(exclude_none=True) for observation in observations.data],
    indent=2,
    default=str,
))

### Reading inputs, outputs, tokens, and cost

The first trace connects the user's question to the policy output, the model prompt, and the final response. This makes it possible to see which information each step received and produced.

The model generation can also record the model name, duration, input tokens, output tokens, and estimated model cost. Cost calculation needs usage information and a matching model price definition. A missing cost value does not mean the request was free.

**Insight:** the final answer shows the result; the trace explains the recorded path to that result. Langfuse's model cost estimate describes model usage, not the application's entire infrastructure bill.

## 4. Debugging a wrong answer

### A successful request can still be incorrect

We now change the policy from 30 days to 3 days. This deliberately introduces incorrect source information while leaving the graph and model unchanged.

The request can complete without an exception. If the model follows the supplied policy, its answer will still be wrong relative to our intended 30-day rule. This is a **quality failure**, even though the application ran successfully.

In [ ]:
POLICY = "Refunds are available within 3 days of purchase."  # Intentional bug.
bug_answer, bug_id = run_demo("What is the refund window?", "wrong-policy")
print(bug_answer)
print("Trace ID:", bug_id)


### Following the evidence to the source

The wrong-policy trace records **3 days** as the output of `read_policy` and carries that value into the model input. If the answer follows that value, the evidence points to the policy source as the cause of the incorrect refund window.

Restoring the intended policy and asking the same question creates a second execution we can compare with the first.

In [ ]:
POLICY = "Refunds are available within 30 days of purchase."
fixed_answer, fixed_id = run_demo("What is the refund window?", "fixed-policy")
print(fixed_answer)
print("Trace ID:", fixed_id)


**Insight:** debugging an LLM application means checking the information passed between steps, not judging only the final response. A model can follow its instructions correctly while the application supplies incorrect evidence.

## 5. Understanding latency and technical failures

### Locating a slow step

The next request adds a deliberate two-second delay to `read_policy`. The trace records where that time was spent. Total request duration also includes the model's variable response time, so total latency alone does not identify the slow component.

In [ ]:
DELAY = 2
try:
    slow_answer, slow_id = run_demo("What is the refund window?", "slow-policy")
    print(slow_answer, "\nTrace ID:", slow_id)
finally:
    DELAY = 0


### Locating a failed step

The next request raises a Python exception in `read_policy`, before the model is called. The notebook catches the expected exception so later sections can continue. The callback records the failure, while the helper still flushes the telemetry.

In [ ]:
FAIL_POLICY = True
try:
    run_demo("What is the refund window?", "policy-error")
except RuntimeError as exc:
    print(exc)
    print("Trace ID:", handler.last_trace_id)
finally:
    FAIL_POLICY = False


**Insight:** duration helps locate a performance problem, while an error identifies a technical failure. The earlier wrong-policy request did not need an exception to be incorrect. Application reliability and answer quality require separate evidence.

## 6. Organizing requests with sessions and context

All requests created by `run_demo()` share the session identifier printed during setup. A session groups related traces; each request still has its own trace.

| Attribute | Meaning in this example |
|---|---|
| Trace name | Identifies the scenario, such as `support-wrong-policy` |
| User ID | Associates the request with `demo-user` |
| Session ID | Groups this notebook execution's related requests |
| Tags | Classify requests with `langfuse-101` and the scenario name |
| Metadata | Adds structured context such as the run identifier and scenario |
| Environment | Identifies these requests as development telemetry |

These attributes help answer questions such as which users experienced an error, which scenarios were slow, or whether a problem appeared in development or production.

**Insight:** a Langfuse session organizes telemetry. It does not give the bot conversational memory. Our graph processes each question independently.

## 7. Managing prompts and versions

### Moving a template into Langfuse

Until now, the prompt template lived in Python. Prompt management stores it in Langfuse, where changes create versions that can be retrieved and linked to model generations.

The first version uses the same instructions as our original template and receives the `production` label. Labels are named pointers to versions; here `production` belongs only to a uniquely named demo prompt.

Langfuse uses `{{policy}}` and `{{question}}` for variables. `get_langchain_prompt()` converts them into LangChain's template syntax. Adding `langfuse_prompt` to the template metadata associates the managed prompt version with the traced generation.

In [ ]:
PROMPT_NAME = f"support-prompt-{RUN}"
v1 = langfuse.create_prompt(
    name=PROMPT_NAME, type="text", labels=["production"],
    prompt="Answer using only this policy: {{policy}}\nQuestion: {{question}}",
)
managed = langfuse.get_prompt(PROMPT_NAME, label="production", cache_ttl_seconds=0)
chain = ChatPromptTemplate.from_template(
    managed.get_langchain_prompt(), metadata={"langfuse_prompt": managed}
) | llm
v1_answer, v1_id = run_demo("What is the refund window?", "prompt-v1")
print(v1_answer, "\nPrompt version:", managed.version)


### Creating a candidate version

The second version asks for one short sentence, the exact number of days, and an explicit admission when the policy cannot answer a question. It receives the `candidate` label, and the application fetches that version for the next request.

Creating a candidate does not promote it to production. The graph stays the same; the selected prompt version changes.

In [ ]:
v2 = langfuse.create_prompt(
    name=PROMPT_NAME, type="text", labels=["candidate"],
    prompt=("Answer in ONE short sentence using only this policy: {{policy}}\n"
            "Include the exact number of days. If the policy does not answer the question, say you do not know.\n"
            "Question: {{question}}"),
)
managed = langfuse.get_prompt(PROMPT_NAME, label="candidate", cache_ttl_seconds=0)
chain = ChatPromptTemplate.from_template(
    managed.get_langchain_prompt(), metadata={"langfuse_prompt": managed}
) | llm
v2_answer, v2_id = run_demo("What is the refund window?", "prompt-v2")
print(v2_answer, "\nPrompt version:", managed.version)


### Prompt caching and the Playground

The demo sets `cache_ttl_seconds=0` so each fetch immediately reads the selected prompt version. In a production application, the SDK's prompt cache can reduce repeated network fetches. This caches the prompt template; provider prompt caching concerns reuse of model input processing and is a different mechanism.

Langfuse's Playground provides a place to try prompt and model changes interactively. A Playground response previews a model call. An application experiment evaluates the surrounding workflow as well.

**Insight:** linking generations to prompt versions makes answers attributable to the instructions that produced them. Clearer instructions are a hypothesis for improvement; version history alone does not prove better quality.

## 8. Recording quality with scores

### A simple human review rubric

Tracing records what happened. A **score** records a judgment about that execution. Scores can come from human review, deterministic code checks, or an LLM judge.

For our refund-window question, the rubric is simple: a correct answer identifies a **30-day refund window**. The next cell displays the bug and fixed answers, collects a human judgment for each, and attaches a Boolean score to its existing trace.

Enter `1` when the answer satisfies the rubric and `0` when it does not. Review the actual response rather than assuming the score from the scenario name.

In [ ]:
print("Bug:", bug_answer)
print("Fix:", fixed_answer)
bug_score = int(input("Does the bug answer correctly state the 30-day window? 0/1: "))
fixed_score = int(input("Does the fixed answer correctly state the 30-day window? 0/1: "))
assert bug_score in (0, 1) and fixed_score in (0, 1)
for trace_id, value in ((bug_id, bug_score), (fixed_id, fixed_score)):
    langfuse.create_score(
        trace_id=trace_id, name="refund-window-correct", value=value,
        data_type="BOOLEAN", comment="Human review: correct answer identifies a 30-day refund window.",
    )
langfuse.flush()
print("Scores attached to existing traces.")


**Insight:** scores add quality information without rerunning the chatbot. A quality judgment needs a clear rubric. A substring check or an LLM judge can automate parts of the process, but each needs validation before its scores can be trusted.

Human feedback can also be collected through trace review and annotation queues. The same principle applies: a recorded response and an explicit criterion form the basis of the judgment.

## 9. Evaluating a version with a dataset and experiment

### Reusable questions and expected answers

A **dataset** stores test cases: inputs and, when available, expected outputs. An **experiment** runs an application version against those cases and records its results.

Our dataset contains two phrasings of the same refund-window question, each with a 30-day expected answer. The experiment reuses the current graph and candidate prompt, making two additional model calls.

There is no judge model or automated evaluator in this example. The experiment records outputs for review; it does not automatically calculate a quality score merely because expected answers are present.

In [ ]:
DATASET_NAME = f"refund-tests-{RUN}"
langfuse.create_dataset(name=DATASET_NAME)
for question in ("What is the refund window?", "How many days do I have to request a refund?"):
    langfuse.create_dataset_item(
        dataset_name=DATASET_NAME, input={"question": question},
        expected_output="Refunds are available within 30 days of purchase.",
    )

def support_task(*, item, **kwargs):
    return graph.invoke(item.input, config={"callbacks": [CallbackHandler()]})["answer"]

dataset = langfuse.get_dataset(DATASET_NAME)
experiment = dataset.run_experiment(
    name=f"candidate-prompt-{RUN}", task=support_task, max_concurrency=1,
)
langfuse.flush()
print(experiment.format())


### Interpreting the experiment

Each result connects the test input, expected output, actual output, and application trace. This ties a quality comparison back to the execution that produced the answer.

A second experiment using the production prompt would allow comparison on the same dataset. Useful comparisons consider correctness, latency, and cost together, with appropriate scoring criteria.

**Insight:** two questions illustrate the workflow but provide little evidence of reliability. A stronger test set includes unsupported questions, boundary cases, and realistic variation. A candidate should be considered better only after reviewing its outputs and relevant metrics.

## 10. Exploring recorded traces with Claude through MCP

### Project access through a standard tool interface

In Claude’s browser session:
1. Open Customize → Connectors → Add custom connector.
2. Name it Langfuse.
3. Enter your project’s endpoint:
   - EU: https://cloud.langfuse.com/api/public/mcp
   - US: https://us.cloud.langfuse.com/api/public/mcp langfuse.com
4. Continue and check for Request headers.
If available, select No sign-in, then add an Authorization header with the value Basic <base64(public_key:secret_key)>. The header supplies authentication despite the “No sign-in” label. Use the below cell to copy the value to your clipboard

"Use Langfuse to list the five most recent root observations in my project. Don’t modify anything." - Try this prompt in the claude chat with Langfuse connector toggled.

In [ ]:
import subprocess

subprocess.run(
    ["clip.exe"],
    input=f"Basic {token}",
    text=True,
    check=True
)

print("Full authorization value copied to clipboard.")